In [6]:
import pandas as pd
import numpy as np

In [16]:
# xây dựng ma trận
data = {
    'User1': [5, 4, np.nan, 2, 1],
    'User2': [4, 5, 3, np.nan, 2],
    'User3': [np.nan, 2, 5, 4, 3],
    'User4': [3, 4, 2, 5, np.nan],
}
df = pd.DataFrame(data, index=['Item1', 'Item2', 'Item3', 'Item4', 'Item5'])

print("Ma trận")
print(df)

df_filled = df.fillna(0)
df_filled


Ma trận
       User1  User2  User3  User4
Item1    5.0    4.0    NaN    3.0
Item2    4.0    5.0    2.0    4.0
Item3    NaN    3.0    5.0    2.0
Item4    2.0    NaN    4.0    5.0
Item5    1.0    2.0    3.0    NaN


,User1,User2,User3,User4
Item1,5.0,4.0,0.0,3.0
Item2,4.0,5.0,2.0,4.0
Item3,0.0,3.0,5.0,2.0
Item4,2.0,0.0,4.0,5.0
Item5,1.0,2.0,3.0,0.0


In [12]:
# tính độ tương đồng giữa 2 user
def pearson_correlation(user1, user2):
    user1_ratings = user1 - user1.mean()
    user2_ratings = user2 - user2.mean()
    return np.dot(user1_ratings, user2_ratings) / (
        np.linalg.norm(user1_ratings) * np.linalg.norm(user2_ratings)
    )

similarity = pearson_correlation(df_filled['User1'], df_filled['User2'])
print(f" Độ tương đồng User1, User2: {similarity:.4f} ")

 Độ tương đồng User1, User2: 0.5265 


In [9]:
# tính độ tương đồng giữa tất cả cặp user
users = df.columns
similarity_matrix = np.zeros((len(users), len(users)))

for i, u1 in enumerate(users):
    for j, u2 in enumerate(users):
        similarity_matrix[i, j] = pearson_correlation(df_filled[u1], df_filled[u2])

similarity_df = pd.DataFrame(similarity_matrix, index=users, columns=users)
print(" Ma trận tương đồng giữa các user ")
print(similarity_df.round(4), "\n")

# Chon k user giong User1 nhat (loai chinh User1 ra)
# >>> Day la phan bai viet bi thieu: phai tu tinh top_similar_users <<<
target_user = 'User1'
k = 2
top_similar_users = (
    similarity_df[target_user]
    .drop(target_user)      # bo chinh minh
    .nlargest(k)            # lay k user co do tuong dong cao nhat
)
print(f" Top {k} user giống {target_user} nhất ")
print(top_similar_users.round(4), "\n")

 Ma trận tương đồng giữa các user 
        User1   User2   User3   User4
User1  1.0000  0.5265 -0.9151  0.4638
User2  0.5265  1.0000 -0.5541 -0.0811
User3 -0.9151 -0.5541  1.0000 -0.0811
User4  0.4638 -0.0811 -0.0811  1.0000 

 Top 2 user giống User1 nhất 
User2    0.5265
User4    0.4638
Name: User1, dtype: float64 



In [10]:
# Dự đoán điểm của một item chưa dc đánh giá.
def predict_rating(item, neighbors):
    """Du doan diem cua target_user cho item, dua tren cac hang xom."""
    neighbor_ratings = df.loc[item, neighbors.index]
    # Chi dung hang xom DA danh gia item nay (bo NaN)
    mask = neighbor_ratings.notna()
    if mask.sum() == 0:
        return np.nan  # khong hang xom nao danh gia -> khong du doan duoc
    sims = neighbors[mask]
    ratings = neighbor_ratings[mask]
    return np.dot(sims, ratings) / sims.sum()

item = 'Item3'
predicted = predict_rating(item, top_similar_users)
print(f"=== Dự đoán điểm của {target_user} cho {item}: {predicted:.4f} ===\n")

=== Dự đoán điểm của User1 cho Item3: 2.5316 ===



In [11]:
def recommend_items(user, n_recommendations=3):
    user_ratings = df[user]
    items_to_predict = user_ratings[user_ratings.isna()].index

    predicted_ratings = []
    for it in items_to_predict:
        p = predict_rating(it, top_similar_users)
        if not np.isnan(p):
            predicted_ratings.append((it, p))

    predicted_ratings.sort(key=lambda x: x[1], reverse=True)
    return predicted_ratings[:n_recommendations]

recommendations = recommend_items(target_user, 3)
print(f"=== Gợi ý cho {target_user} ===")
for it, score in recommendations:
    print(f"  {it}: điểm dự đoán = {score:.4f}")


=== Gợi ý cho User1 ===
  Item3: điểm dự đoán = 2.5316
